<a href="https://colab.research.google.com/github/Marwan77770/Marwan_FlyRank/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Marwan77770/Marwan_FlyRank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
from google.colab import userdata
from huggingface_hub import login

token = userdata.get("HF_TOKEN")
login(token=token)

print("Hugging Face login successful.")

Hugging Face login successful.


In [ ]:
from huggingface_hub import list_repo_files

repo_id = "FlyRank/internship-warehouse"

files = list_repo_files(
    repo_id=repo_id,
    repo_type="dataset"
)

for f in files:
    print(f)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

In [ ]:
from huggingface_hub import hf_hub_download
import pandas as pd

repo_id = "FlyRank/internship-warehouse"

file_path = hf_hub_download(
    repo_id=repo_id,
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset"
)

march_df = pd.read_parquet(file_path)

print("Rows:", len(march_df))
print("Columns:", len(march_df.columns))
print("\nColumns:")
print(march_df.columns.tolist())

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Rows: 9841378
Columns: 30

Columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


## 1. Unit of analysis + time window

Unit of analysis: One row represents one content page for one client on one report date.

Time window: March 2026, from 2026-03-01 to 2026-03-31.

The March 2026 slice contains 9,841,378 rows, 55 clients, and 331,437 unique content pages. The verified grain is the combination of client_hash_id, content_hash_id, and report_date.

What I rank: I rank content pages for human review based on observed search, content, and engagement signals available by the decision moment.

What I deliberately exclude: Any label-derived or future-outcome field is excluded from the feature set to prevent target leakage.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### 2. Fields: feature / label / context / excluded

#### Features

1. `march_impressions` — total Google Search impressions observed during March 2026.
   **Available when:** After the March observation window closes, because it uses only March daily observations.

2. `avg_search_position` — mean observed Google Search position during March 2026.
   **Available when:** After the March observation window closes, using only position observations available during March.

3. `march_sessions` — total Google Analytics sessions observed during March 2026.
   **Available when:** After the March observation window closes, using only March session observations.

4. `engagement_rate` — engaged sessions divided by sessions during March 2026.
   **Available when:** After the March observation window closes, because it is calculated only from March sessions and engaged-session observations.

5. `march_scroll_events` — total scroll events observed during March 2026.
   **Available when:** After the March observation window closes, using only March scroll-event observations.

#### Label / Proxy

The ranking proxy is `is_declining_proxy`.

A page is labeled 1 when its observed April 2026 Google Search clicks are lower than its observed March 2026 Google Search clicks:

`April clicks < March clicks`

This is a proxy for observed directional decline. It is not a guarantee of future performance or an explanation of why performance changed.

#### Context

* `content_hash_id` — identifies the content page being ranked.
* `client_hash_id` — identifies the client associated with the page.
* `report_date` — identifies the daily observation date.

#### Excluded

* `march_clicks` — deliberately excluded because March clicks are directly used to construct the proxy target, creating target leakage.
* April 2026 performance fields — excluded from features because April is used only to construct the proxy outcome.
* Any label-derived field — excluded to prevent target leakage.
* `_sample` — excluded from development because it represents the sealed final-month sample.


In [ ]:
print("Content dimension columns:")
print(pd.read_parquet(
    hf_hub_download(
        repo_id=repo_id,
        filename="dim_content.parquet",
        repo_type="dataset"
    )
).columns.tolist())

Content dimension columns:


dim_content.parquet: reconstructing file:   0%|          |  0.00B / 19.6MB            

dim_content.parquet: downloading bytes:           |  0.00B            

['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']


In [ ]:
content_df = pd.read_parquet(
    hf_hub_download(
        repo_id=repo_id,
        filename="dim_content.parquet",
        repo_type="dataset"
    )
)

print("Rows:", len(content_df))
print("Columns:", len(content_df.columns))

print("\nPotential outcome / trend fields:")
for col in content_df.columns:
    if any(word in col.lower() for word in [
        "trend", "declin", "refresh", "performance",
        "updated", "optimized", "eligible"
    ]):
        print(col)

Rows: 519606
Columns: 26

Potential outcome / trend fields:
content_updated_date
last_optimized_date
optimization_eligible_date


In [ ]:
query90_df = pd.read_parquet(
    hf_hub_download(
        repo_id=repo_id,
        filename="fact_content_query_90d.parquet",
        repo_type="dataset"
    )
)

print("Rows:", len(query90_df))
print("Columns:", len(query90_df.columns))

print("\nColumns:")
print(query90_df.columns.tolist())

fact_content_query_90d.parquet: reconstructing file:   0%|          |  0.00B / 60.7MB            

fact_content_query_90d.parquet: downloading bytes:           |  0.00B            

Rows: 2414248
Columns: 21

Columns:
['client_hash_id', 'content_hash_id', 'query_hash_id', 'query_char_count', 'query_token_count', 'window_start', 'window_end', 'impressions_90d', 'clicks_90d', 'impressions_last30', 'clicks_last30', 'impressions_prev30', 'clicks_prev30', 'avg_position_90d', 'avg_position_last30', 'avg_position_prev30', 'content_total_impressions_90d', 'content_visible_query_count', 'rare_query_count', 'rare_impressions_share', 'anonymized_impressions_share']


In [ ]:
print("Window start:", query90_df["window_start"].min())
print("Window end:", query90_df["window_end"].max())

print("\nUnique window pairs:")
print(
    query90_df[["window_start", "window_end"]]
    .drop_duplicates()
    .sort_values(["window_start", "window_end"])
    .tail(10)
)

Window start: 2026-04-02
Window end: 2026-06-30

Unique window pairs:
  window_start  window_end
0   2026-04-02  2026-06-30


In [ ]:
daily_counts = (
    march_df.groupby("report_date")
    .size()
    .reset_index(name="rows")
)

print(daily_counts.to_string(index=False))

report_date   rows
 2026-03-01 275874
 2026-03-02 276269
 2026-03-03 311676
 2026-03-04 311675
 2026-03-05 311676
 2026-03-06 312187
 2026-03-07 312387
 2026-03-08 313374
 2026-03-09 313874
 2026-03-10 314047
 2026-03-11 314232
 2026-03-12 317742
 2026-03-13 317900
 2026-03-14 319584
 2026-03-15 319758
 2026-03-16 319924
 2026-03-17 320048
 2026-03-18 320682
 2026-03-19 323054
 2026-03-20 323738
 2026-03-21 324296
 2026-03-22 324476
 2026-03-23 324899
 2026-03-24 325117
 2026-03-25 325605
 2026-03-26 326039
 2026-03-27 326193
 2026-03-28 326193
 2026-03-29 326193
 2026-03-30 331230
 2026-03-31 331436


In [ ]:
feature_candidates = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions"
]

print(march_df[feature_candidates].isna().sum())

gsc_impressions               0
gsc_clicks                    0
gsc_avg_position        6230317
ga4_sessions            3018741
ga4_engaged_sessions    3018741
dtype: int64


In [ ]:
scroll_feature = (
    march_df.groupby("content_hash_id")["scroll_events"]
    .sum()
    .rename("march_scroll_events")
)

print("Pages with scroll feature:", len(scroll_feature))
print("Missing values:", scroll_feature.isna().sum())
display(scroll_feature.head())

Pages with scroll feature: 331437
Missing values: 0


,march_scroll_events
content_hash_id,
content_000005d4ced12088,0.0
content_00001e488b74b799,0.0
content_00007bd2985b77c3,0.0
content_00008950670cb6b5,1.0
content_0000a348850eb1fc,1.0


In [ ]:
content_client_counts = (
    march_df.groupby("content_hash_id")["client_hash_id"]
    .nunique()
)

print("Content pages:", len(content_client_counts))
print("Pages linked to exactly 1 client:",
      (content_client_counts == 1).sum())
print("Pages linked to multiple clients:",
      (content_client_counts > 1).sum())
print("Maximum clients per content page:",
      content_client_counts.max())

Content pages: 331437
Pages linked to exactly 1 client: 331437
Pages linked to multiple clients: 0
Maximum clients per content page: 1


In [ ]:
feature_df = (
    march_df.groupby("content_hash_id")
    .agg(
        march_impressions=("gsc_impressions", "sum"),
        march_clicks=("gsc_clicks", "sum"),
        avg_search_position=("gsc_avg_position", "mean"),
        march_sessions=("ga4_sessions", "sum"),
        march_engaged_sessions=("ga4_engaged_sessions", "sum")
    )
    .reset_index()
)

feature_df["engagement_rate"] = (
    feature_df["march_engaged_sessions"]
    / feature_df["march_sessions"].replace(0, pd.NA)
)

feature_df = feature_df[
    [
        "content_hash_id",
        "march_impressions",
        "march_clicks",
        "avg_search_position",
        "march_sessions",
        "engagement_rate"
    ]
]

print("Feature frame shape:", feature_df.shape)
display(feature_df.head())

Feature frame shape: (331437, 6)


,content_hash_id,march_impressions,march_clicks,avg_search_position,march_sessions,engagement_rate
0,content_000005d4ced12088,86,0,72.854861,0.0,<NA>
1,content_00001e488b74b799,0,0,NaN,0.0,<NA>
2,content_00007bd2985b77c3,47,0,5.269565,0.0,<NA>
3,content_00008950670cb6b5,0,0,NaN,2.0,0.5
4,content_0000a348850eb1fc,0,0,NaN,1.0,0.0


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
#Grain
print("Unique dates:", march_df["report_date"].nunique())
print("Unique clients:", march_df["client_hash_id"].nunique())
print("Unique content pages:", march_df["content_hash_id"].nunique())

print(
    "Unique client-content-date rows:",
    march_df[["client_hash_id", "content_hash_id", "report_date"]].drop_duplicates().shape[0]
)

print("Total rows:", len(march_df))

Unique dates: 31
Unique clients: 55
Unique content pages: 331437
Unique client-content-date rows: 9841378
Total rows: 9841378


In [ ]:
print("March 2026 row count:", len(march_df))
print("Date span:", march_df["report_date"].min(), "to", march_df["report_date"].max())

March 2026 row count: 9841378
Date span: 2026-03-01 to 2026-03-31


In [ ]:
print("GSC available:")
print(march_df["gsc_data_available"].value_counts(dropna=False))

print("\nGA4 available:")
print(march_df["ga4_data_available"].value_counts(dropna=False))

print("\nRows with GSC available (IS TRUE):")
print(march_df["gsc_data_available"].eq(True).sum())

print("\nRows with GA4 available (IS TRUE):")
print(march_df["ga4_data_available"].eq(True).sum())

GSC available:
gsc_data_available
False    6230317
True     3611061
Name: count, dtype: int64

GA4 available:
ga4_data_available
False    6408671
None     3018741
True      413966
Name: count, dtype: int64

Rows with GSC available (IS TRUE):
3611061

Rows with GA4 available (IS TRUE):
413966


## 4. Data limits


This data supports observed, directional analysis of content-page performance, but it cannot establish causality.

The March 2026 daily performance table contains pages with different levels of data availability. Google Search Console and Google Analytics 4 are not available for every row, so features based on these sources can have missing values.

The dataset also does not directly tell us why a page's performance changed. A lower number of impressions, clicks, or sessions may have multiple possible causes that are not observable in these tables.

The March slice is useful for development, but future-month performance must not be used as an input feature because it would not be available at the decision moment.

The sealed `_sample` table is excluded from feature development and label design to avoid using final-test information during development.

The resulting ranking should therefore be treated as **decision support for human content review**, not as proof that a page will decline or that a specific refresh action will improve performance.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [ ]:
feb_path = hf_hub_download(
    repo_id=repo_id,
    filename="fact_content_daily_performance/month=2026-02/data_0.parquet",
    repo_type="dataset"
)

feb_df = pd.read_parquet(feb_path)

print("February rows:", len(feb_df))
print("Date span:", feb_df["report_date"].min(), "to", feb_df["report_date"].max())

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 89.0MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

February rows: 7355108
Date span: 2026-02-01 to 2026-02-28


In [ ]:
feb_clicks = (
    feb_df.groupby("content_hash_id")["gsc_clicks"]
    .sum()
    .rename("feb_clicks")
)

mar_clicks = (
    march_df.groupby("content_hash_id")["gsc_clicks"]
    .sum()
    .rename("mar_clicks")
)

proxy_df = pd.concat([feb_clicks, mar_clicks], axis=1).fillna(0)

proxy_df["is_declining_proxy"] = (
    proxy_df["mar_clicks"] < proxy_df["feb_clicks"]
).astype(int)

print("Pages:", len(proxy_df))
print("Declining proxy:", proxy_df["is_declining_proxy"].sum())
print("Non-declining proxy:", (proxy_df["is_declining_proxy"] == 0).sum())

display(proxy_df.head())

Pages: 349411
Declining proxy: 26181
Non-declining proxy: 323230


,feb_clicks,mar_clicks,is_declining_proxy
content_hash_id,,,
content_000005d4ced12088,0.0,0.0,0
content_00001e488b74b799,0.0,0.0,0
content_00007bd2985b77c3,0.0,0.0,0
content_00008950670cb6b5,0.0,0.0,0
content_0000a348850eb1fc,0.0,0.0,0


In [ ]:
apr_path = hf_hub_download(
    repo_id=repo_id,
    filename="fact_content_daily_performance/month=2026-04/data_0.parquet",
    repo_type="dataset"
)

apr_df = pd.read_parquet(apr_path)

print("April rows:", len(apr_df))
print("Date span:", apr_df["report_date"].min(), "to", apr_df["report_date"].max())

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  134MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

April rows: 10424730
Date span: 2026-04-01 to 2026-04-30


In [ ]:
apr_clicks = (
    apr_df.groupby("content_hash_id")["gsc_clicks"]
    .sum()
    .rename("apr_clicks")
)

mar_clicks = (
    march_df.groupby("content_hash_id")["gsc_clicks"]
    .sum()
    .rename("mar_clicks")
)

honest_proxy_df = pd.concat(
    [mar_clicks, apr_clicks],
    axis=1
).fillna(0)

honest_proxy_df["is_declining_proxy"] = (
    honest_proxy_df["apr_clicks"] < honest_proxy_df["mar_clicks"]
).astype(int)

print("Pages:", len(honest_proxy_df))
print("Declining proxy:", honest_proxy_df["is_declining_proxy"].sum())
print("Non-declining proxy:",
      (honest_proxy_df["is_declining_proxy"] == 0).sum())

display(honest_proxy_df.head())

Pages: 362173
Declining proxy: 45102
Non-declining proxy: 317071


,mar_clicks,apr_clicks,is_declining_proxy
content_hash_id,,,
content_000005d4ced12088,0.0,0.0,0
content_00001e488b74b799,0.0,0.0,0
content_00007bd2985b77c3,0.0,0.0,0
content_00008950670cb6b5,0.0,0.0,0
content_0000a348850eb1fc,0.0,0.0,0


In [ ]:
model_df = feature_df.merge(
    honest_proxy_df[["is_declining_proxy"]],
    left_on="content_hash_id",
    right_index=True,
    how="inner"
)

print("Model dataset shape:", model_df.shape)
print("Target positive:", model_df["is_declining_proxy"].sum())
print("Target negative:",
      (model_df["is_declining_proxy"] == 0).sum())

Model dataset shape: (331437, 7)
Target positive: 45102
Target negative: 286335


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

leak_df = model_df.copy()

feature_cols = [
    "march_impressions",
    "march_clicks",
    "avg_search_position",
    "march_sessions",
    "engagement_rate"
]

X = leak_df[feature_cols].copy()

for col in feature_cols:
    X[col] = pd.to_numeric(X[col], errors="coerce")

y = leak_df["is_declining_proxy"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

imputer = SimpleImputer(strategy="median")

X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

leak_model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

leak_model.fit(X_train, y_train)

leak_score = roc_auc_score(
    y_test,
    leak_model.predict_proba(X_test)[:, 1]
)

print("LEAKED MODEL")
print("ROC AUC:", round(leak_score, 4))

LEAKED MODEL
ROC AUC: 0.9677


In [ ]:
honest_features = [
    "march_impressions",
    "avg_search_position",
    "march_sessions",
    "engagement_rate"
]

X_honest = model_df[honest_features].copy()

for col in honest_features:
    X_honest[col] = pd.to_numeric(X_honest[col], errors="coerce")

y = model_df["is_declining_proxy"]

X_train, X_test, y_train, y_test = train_test_split(
    X_honest,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

imputer = SimpleImputer(strategy="median")

X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

honest_model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

honest_model.fit(X_train, y_train)

honest_score = roc_auc_score(
    y_test,
    honest_model.predict_proba(X_test)[:, 1]
)

print("HONEST MODEL")
print("ROC AUC:", round(honest_score, 4))

HONEST MODEL
ROC AUC: 0.9238


### Deliberate Leakage Trap

I deliberately introduced `march_clicks` as a feature in the first experiment.

This feature is leaked because the proxy target is defined using the comparison:

`April clicks < March clicks`

Therefore, March clicks are directly part of the target definition. Using `march_clicks` as an input allows the model to access information that is also used to construct the proxy.

The leaked model achieved:

**ROC AUC = 0.9677**

I then removed `march_clicks` from the feature set and retrained the same model using the remaining four features.

The honest model achieved:

**ROC AUC = 0.9238**

The decrease from 0.9677 to 0.9238 demonstrates why target-derived fields must be excluded from the feature set.

**Final decision:** `march_clicks` is excluded from the honest feature set. The honest ROC AUC of **0.9238** is retained as the valid experimental result.

This experiment is a leakage check, not evidence that the model will achieve the same performance on future unseen data.


In [ ]:
feature_df = feature_df.merge(
    scroll_feature,
    on="content_hash_id",
    how="left"
)

final_features = [
    "content_hash_id",
    "march_impressions",
    "avg_search_position",
    "march_sessions",
    "engagement_rate",
    "march_scroll_events"
]

feature_df = feature_df[final_features]

print("Final feature frame shape:", feature_df.shape)
print("Final features:")
print(feature_df.columns.tolist())

Final feature frame shape: (331437, 6)
Final features:
['content_hash_id', 'march_impressions', 'avg_search_position', 'march_sessions', 'engagement_rate', 'march_scroll_events']


### Self-Check

* [x] Five contract answers are documented: unit of analysis, tables, time window, ranking target/proxy, and deliberate exclusion.
* [x] Three verification queries were executed with visible outputs.
* [x] Grain was verified as one client-content-date observation per row.
* [x] March 2026 was verified as the development time window, covering 2026-03-01 to 2026-03-31.
* [x] Data availability was checked for Google Search Console and Google Analytics 4 using boolean availability checks.
* [x] A five-feature frame was built at content-page level.
* [x] Each feature has an availability-at-decision-moment statement.
* [x] A deliberate target-leakage experiment was performed.
* [x] The leaked model achieved ROC AUC = 0.9677.
* [x] The leaked feature `march_clicks` was removed.
* [x] The honest model achieved ROC AUC = 0.9238.
* [x] One named limitation is documented: the data supports observed directional analysis but does not establish causality.
* [x] No client names, private queries, or sensitive credentials are included.
* [x] The notebook uses careful language such as observed, measured, directional, proxy, and decision support.
